In [1]:
# ============================================================
# CELL 1: Check GPU
# ============================================================
!nvidia-smi

import sys
import torch

print(f"\n{'='*70}")
print("SYSTEM INFORMATION")
print(f"{'='*70}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU! Enable GPU in Runtime → Change runtime type → GPU")

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Colab: {IN_COLAB}")
print(f"{'='*70}")

/bin/bash: line 1: nvidia-smi: command not found



SYSTEM INFORMATION
Python: 3.12.3
PyTorch: 2.7.1+cu118
CUDA Available: False
⚠️ WARNING: No GPU! Enable GPU in Runtime → Change runtime type → GPU
Running in Colab: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 2: Install Packages (run once, then restart runtime)
# ============================================================
!pip uninstall bitsandbytes transformers peft accelerate -y -q

!pip install -q transformers==4.44.0
!pip install -q peft==0.7.1
!pip install -q accelerate==0.33.0
!pip install -q datasets==2.20.0
!pip install -q scipy scikit-learn

print("✅ All packages installed!")
print("⚠️  Now go to Runtime → Restart session, then continue from Cell 3")

✅ All packages installed!
⚠️  Now go to Runtime → Restart session, then continue from Cell 3


In [12]:
# ============================================================
# CELL 3: Verify Installations
# ============================================================
import importlib.metadata
import transformers
import datasets
import peft

print("\n📦 Package Versions:")
print(f"  Transformers: {transformers.__version__}")
print(f"  Datasets:     {datasets.__version__}")
print(f"  PEFT:         {peft.__version__}")
print("\n✅ All imports successful!")


📦 Package Versions:
  Transformers: 4.44.0
  Datasets:     2.20.0
  PEFT:         0.7.1

✅ All imports successful!


In [13]:
# ============================================================
# CELL 4: Upload Dataset
# ============================================================
from google.colab import files
import os
import json

os.makedirs('data', exist_ok=True)

print(f"{'='*70}")
print("UPLOAD YOUR DATASET")
print(f"{'='*70}")
print("\nExpected JSONL format:")
print('{"instruction": "question", "input": "context", "output": "answer"}')
print("\n📁 Click 'Choose Files' to upload train.jsonl:\n")

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(('.jsonl', '.json')):
        os.rename(filename, 'data/train.jsonl')
        print(f"\n✅ Uploaded: {filename} → data/train.jsonl")

if os.path.exists('data/train.jsonl'):
    print("\n📄 Preview (first 3 lines):")
    print("-" * 70)
    !head -3 data/train.jsonl
    print("-" * 70)

    with open('data/train.jsonl', 'r') as f:
        total = sum(1 for _ in f)
    print(f"\n📊 Total training samples: {total}")
else:
    print("\n❌ Error: Dataset not found!")

UPLOAD YOUR DATASET

Expected JSONL format:
{"instruction": "question", "input": "context", "output": "answer"}

📁 Click 'Choose Files' to upload train.jsonl:



Saving train.jsonl to train.jsonl

✅ Uploaded: train.jsonl → data/train.jsonl

📄 Preview (first 3 lines):
----------------------------------------------------------------------
{"instruction": "Where is the email for Ariana?", "input": "", "output": "The email is located in marcia73@example.com."}
{"instruction": "Extract the email and phone number from this text.", "input": "Customer information: email is jennabarry@figueroa.com, phone is 643.470.0320x81435", "output": "{\n  \"email\": \"jennabarry@figueroa.com\",\n  \"phone\": \"643.470.0320x81435\"\n}"}
{"instruction": "Analyze the geographical and market implications of Leroy's location.", "input": "", "output": "Leroy is located in Kaufmanburgh, Eritrea. This location implies: timezone considerations for optimal communication timing, potential shipping and logistics requirements, regional market preferences and cultural nuances, and localized marketing strategies. Consider these factors when planning engagement."}
----------------

In [14]:
# ============================================================
# CELL 5: Create Validation Split
# ============================================================
import json

print("Creating validation set...")

with open('data/train.jsonl', 'r') as f:
    all_data = [json.loads(line) for line in f]

total_samples = len(all_data)
val_size = min(20, max(5, int(total_samples * 0.1)))

print(f"\nDataset Split:")
print(f"  Total:      {total_samples}")
print(f"  Validation: {val_size}")
print(f"  Training:   {total_samples - val_size}")

with open('data/val.jsonl', 'w') as f:
    for sample in all_data[:val_size]:
        f.write(json.dumps(sample) + '\n')

with open('data/train_split.jsonl', 'w') as f:
    for sample in all_data[val_size:]:
        f.write(json.dumps(sample) + '\n')

print("\n✅ Split complete!")
print("\n📋 Validation sample:")
!head -1 data/val.jsonl | python -m json.tool

Creating validation set...

Dataset Split:
  Total:      1016
  Validation: 20
  Training:   996

✅ Split complete!

📋 Validation sample:
{
    "instruction": "Where is the email for Ariana?",
    "input": "",
    "output": "The email is located in marcia73@example.com."
}


In [15]:
# ============================================================
# CELL 6: Configuration
# ============================================================
CONFIG = {
    "model_name":                  "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "lora_r":                      8,
    "lora_alpha":                  16,
    "lora_dropout":                0.1,
    "target_modules":              ["q_proj", "k_proj", "v_proj", "o_proj"],
    "learning_rate":               2e-4,
    "num_epochs":                  3,
    "batch_size":                  4,
    "gradient_accumulation_steps": 4,
    "max_seq_length":              512,
    "warmup_steps":                50,
    "weight_decay":                0.01,
    "fp16":                        True,
    "logging_steps":               5,
    "eval_steps":                  50,
    "save_steps":                  100,
}

print(f"{'='*70}")
print("TRAINING CONFIGURATION")
print(f"{'='*70}")
for key, value in CONFIG.items():
    print(f"  {key:30s}: {value}")
print(f"{'='*70}")

effective_batch = CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"\n📊 Effective batch size: {effective_batch}")

TRAINING CONFIGURATION
  model_name                    : TinyLlama/TinyLlama-1.1B-Chat-v1.0
  lora_r                        : 8
  lora_alpha                    : 16
  lora_dropout                  : 0.1
  target_modules                : ['q_proj', 'k_proj', 'v_proj', 'o_proj']
  learning_rate                 : 0.0002
  num_epochs                    : 3
  batch_size                    : 4
  gradient_accumulation_steps   : 4
  max_seq_length                : 512
  warmup_steps                  : 50
  weight_decay                  : 0.01
  fp16                          : True
  logging_steps                 : 5
  eval_steps                    : 50
  save_steps                    : 100

📊 Effective batch size: 16


In [16]:
# ============================================================
# CELL 7: Load Tokenizer
# ============================================================
from transformers import AutoTokenizer

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print("  ⚙️ Set pad_token = eos_token")

tokenizer.padding_side = "right"

print(f"\n✅ Tokenizer loaded!")
print(f"  Vocab size: {len(tokenizer):,}")
print(f"  Pad token:  '{tokenizer.pad_token}'")
print(f"  EOS token:  '{tokenizer.eos_token}'")

test = tokenizer("Hello, world!")
print(f"\n🧪 Test: {test['input_ids'][:10]}...")

Loading tokenizer...

✅ Tokenizer loaded!
  Vocab size: 32,000
  Pad token:  '</s>'
  EOS token:  '</s>'

🧪 Test: [1, 15043, 29892, 3186, 29991]...


In [17]:
# ============================================================
# CELL 8: Load Model in FP16
# ============================================================
from transformers import AutoModelForCausalLM
import torch

print("Loading model in FP16...")
print(f"Model: {CONFIG['model_name']}\n")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("✅ Model loaded!\n")

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Model Statistics:")
print(f"  Total parameters:  {total_params:,}")
print(f"  Model size (FP16): ~{total_params * 2 / 1e9:.2f} GB")

if torch.cuda.is_available():
    print(f"  GPU Memory used:   {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  GPU Memory total:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Loading model in FP16...
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0



config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!

📊 Model Statistics:
  Total parameters:  1,100,048,384
  Model size (FP16): ~2.20 GB
  GPU Memory used:   2.20 GB
  GPU Memory total:  15.6 GB


In [18]:
# ============================================================
# CELL 9: Add LoRA Adapters
# ============================================================
from peft import LoraConfig, get_peft_model

print("Adding LoRA adapters...\n")

model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Cast LoRA params to float32 to avoid FP16 gradient issues
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
trainable_pct    = (trainable_params / total_params) * 100

print("✅ LoRA adapters added!\n")
print(f"📊 Parameter Efficiency:")
print(f"  Trainable:  {trainable_params:,}")
print(f"  Total:      {total_params:,}")
print(f"  Percentage: {trainable_pct:.4f}%")
print(f"\n  🎯 Only {trainable_pct:.2f}% of parameters will be trained!")

Adding LoRA adapters...

✅ LoRA adapters added!

📊 Parameter Efficiency:
  Trainable:  2,252,800
  Total:      1,102,301,184
  Percentage: 0.2044%

  🎯 Only 0.20% of parameters will be trained!


In [19]:
# ============================================================
# CELL 10: Load Datasets
# ============================================================
from datasets import load_dataset

print("Loading datasets...\n")

train_dataset = load_dataset("json", data_files="data/train_split.jsonl", split="train")
eval_dataset  = load_dataset("json", data_files="data/val.jsonl",         split="train")

print("✅ Datasets loaded!")
print(f"  Training:   {len(train_dataset)} samples")
print(f"  Validation: {len(eval_dataset)} samples")

print(f"\n📋 Training sample:")
sample = train_dataset[0]
for key, value in sample.items():
    display_val = str(value)[:80] + "..." if len(str(value)) > 80 else str(value)
    print(f"  {key}: {display_val}")

Loading datasets...



Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Datasets loaded!
  Training:   996 samples
  Validation: 20 samples

📋 Training sample:
  instruction: Analyze the profile of Sylvia and provide business insights.
  input: 
  output: Profile analysis for Sylvia: .


In [20]:
# ============================================================
# CELL 11: Tokenize Datasets
# ============================================================
def format_text(sample):
    instruction = sample["instruction"]
    input_text  = sample.get("input", "")
    output      = sample["output"]

    if input_text:
        text = f"<|im_start|>user\n{instruction}\n\nContext: {input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    else:
        text = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    return text

def tokenize_function(sample):
    tokenized = tokenizer(
        format_text(sample),
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        padding="max_length",
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing datasets...\n")

tokenized_train = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training"
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    remove_columns=eval_dataset.column_names,
    desc="Tokenizing validation"
)

tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

print("✅ Tokenization complete!")
print(f"  Train: {len(tokenized_train)} samples")
print(f"  Eval:  {len(tokenized_eval)} samples")
print(f"\n🔍 Tokenized example:")
print(f"  Sequence length: {len(tokenized_train[0]['input_ids'])}")
print(f"  First 15 tokens: {tokenized_train[0]['input_ids'][:15]}")

Tokenizing datasets...



Tokenizing training:   0%|          | 0/996 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/20 [00:00<?, ? examples/s]

✅ Tokenization complete!
  Train: 996 samples
  Eval:  20 samples

🔍 Tokenized example:
  Sequence length: 512
  First 15 tokens: tensor([    1,   529, 29989,   326, 29918,  2962, 29989, 29958,  1792,    13,
         2744, 14997,   911,   278,  8722])


In [21]:
# ============================================================
# CELL 12: Setup Training
# ============================================================
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

print("Configuring training...\n")

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    fp16=False,                        # disabled — LoRA uses float32 params
    bf16=False,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # suppress warning
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("✅ Trainer configured!\n")

steps_per_epoch = len(tokenized_train) // (CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"])
total_steps     = steps_per_epoch * CONFIG["num_epochs"]

print(f"📊 Training Plan:")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps:     {total_steps}")
print(f"  Eval every:      {CONFIG['eval_steps']} steps")
print(f"  Estimated time:  ~{total_steps * 0.5 / 60:.0f}-{total_steps * 1 / 60:.0f} minutes")

Configuring training...

✅ Trainer configured!

📊 Training Plan:
  Steps per epoch: 62
  Total steps:     186
  Eval every:      50 steps
  Estimated time:  ~2-3 minutes


In [22]:
# ============================================================
# CELL 13: START TRAINING 🚀
# ============================================================
from datetime import datetime

print(f"{'='*70}")
print("STARTING TRAINING")
print(f"{'='*70}\n")

start_time = datetime.now()
print(f"⏰ Start time: {start_time.strftime('%H:%M:%S')}\n")

train_result = trainer.train()

end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*70}")
print("✅ TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"⏰ End time:     {end_time.strftime('%H:%M:%S')}")
print(f"⏱️  Duration:    {duration}")
print(f"📉 Final Loss:  {train_result.training_loss:.4f}")
print(f"🔢 Total Steps: {train_result.global_step}")
print(f"{'='*70}")

STARTING TRAINING

⏰ Start time: 17:54:49



Step,Training Loss,Validation Loss
50,1.166500,1.001455
100,0.513500,0.563735
150,0.470800,0.521997



✅ TRAINING COMPLETE!
⏰ End time:     18:07:11
⏱️  Duration:    0:12:21.961207
📉 Final Loss:  0.9485
🔢 Total Steps: 186


In [23]:
# ============================================================
# CELL 14: Save Adapter
# ============================================================
import os

print("Saving fine-tuned adapter...\n")

output_dir = "./lora_adapter"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Adapter saved to: {output_dir}")
print("\n📁 Saved files:")
!ls -lh {output_dir}

total_size = sum(
    os.path.getsize(os.path.join(output_dir, f))
    for f in os.listdir(output_dir)
    if os.path.isfile(os.path.join(output_dir, f))
)
print(f"\n💾 Total size: {total_size / 1e6:.2f} MB")

Saving fine-tuned adapter...

✅ Adapter saved to: ./lora_adapter

📁 Saved files:
total 11M
-rw-r--r-- 1 root root  613 Mar  1 18:08 adapter_config.json
-rw-r--r-- 1 root root 8.7M Mar  1 18:08 adapter_model.safetensors
-rw-r--r-- 1 root root 5.0K Mar  1 18:08 README.md
-rw-r--r-- 1 root root  551 Mar  1 18:08 special_tokens_map.json
-rw-r--r-- 1 root root 1.4K Mar  1 18:08 tokenizer_config.json
-rw-r--r-- 1 root root 1.8M Mar  1 18:08 tokenizer.json
-rw-r--r-- 1 root root 489K Mar  1 18:08 tokenizer.model

💾 Total size: 11.38 MB


In [24]:
# ============================================================
# CELL 15: Generate Training Report
# ============================================================
report = f"""
{'='*70}
LORA FINE-TUNING REPORT
{'='*70}

📅 Date:  {end_time.strftime('%Y-%m-%d %H:%M:%S')}
🤖 Model: {CONFIG['model_name']}

CONFIGURATION
{'-'*70}
LoRA Parameters:
  • Rank (r):   {CONFIG['lora_r']}
  • Alpha:      {CONFIG['lora_alpha']}
  • Dropout:    {CONFIG['lora_dropout']}
  • Targets:    {', '.join(CONFIG['target_modules'])}

Training Parameters:
  • Learning rate:          {CONFIG['learning_rate']}
  • Epochs:                 {CONFIG['num_epochs']}
  • Batch size:             {CONFIG['batch_size']}
  • Gradient accumulation:  {CONFIG['gradient_accumulation_steps']}
  • Max sequence length:    {CONFIG['max_seq_length']}
  • Precision:              FP32 (LoRA params)

RESULTS
{'-'*70}
  • Duration:         {duration}
  • Final loss:       {train_result.training_loss:.4f}
  • Total steps:      {train_result.global_step}
  • Trainable params: {trainable_params:,} / {total_params:,} ({trainable_pct:.4f}%)

Dataset:
  • Training samples:   {len(train_dataset)}
  • Validation samples: {len(eval_dataset)}

OUTPUT
{'-'*70}
  📁 Adapter: {output_dir}
  💾 Size:    {total_size / 1e6:.2f} MB

{'='*70}
"""

print(report)

with open("training_report.txt", "w") as f:
    f.write(report)

print("✅ Report saved to: training_report.txt")


LORA FINE-TUNING REPORT

📅 Date:  2026-03-01 18:07:11
🤖 Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0

CONFIGURATION
----------------------------------------------------------------------
LoRA Parameters:
  • Rank (r):   8
  • Alpha:      16
  • Dropout:    0.1
  • Targets:    q_proj, k_proj, v_proj, o_proj

Training Parameters:
  • Learning rate:          0.0002
  • Epochs:                 3
  • Batch size:             4
  • Gradient accumulation:  4
  • Max sequence length:    512
  • Precision:              FP32 (LoRA params)

RESULTS
----------------------------------------------------------------------
  • Duration:         0:12:21.961207
  • Final loss:       0.9485
  • Total steps:      186
  • Trainable params: 2,252,800 / 1,102,301,184 (0.2044%)

Dataset:
  • Training samples:   996
  • Validation samples: 20

OUTPUT
----------------------------------------------------------------------
  📁 Adapter: ./lora_adapter
  💾 Size:    11.38 MB


✅ Report saved to: training_report.txt


In [25]:
# ============================================================
# CELL 16: Load Model for Testing
# ============================================================
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

print("Loading model for testing...\n")

base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

test_model = PeftModel.from_pretrained(base_model, "./lora_adapter")
test_model.eval()

print("✅ Model loaded and ready for inference!")

Loading model for testing...

✅ Model loaded and ready for inference!


In [26]:
# ============================================================
# CELL 17: Generation Function
# ============================================================
def generate_response(instruction, max_new_tokens=150, temperature=0.7):
    prompt = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(test_model.device)

    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant\n")[-1].strip()
    response = response.split("<|im_end|>")[0].strip()
    return response

print("✅ Generation function ready!")
print("\nUsage: generate_response('Your question here')")

✅ Generation function ready!

Usage: generate_response('Your question here')


In [27]:
# ============================================================
# CELL 18: Run Tests
# ============================================================
test_prompts = [
    "What is artificial intelligence?",
    "Explain machine learning in simple terms.",
    "What is the difference between AI and ML?",
    "How does deep learning work?",
    "What is Python programming?",
]

print(f"{'='*70}")
print("TESTING FINE-TUNED MODEL")
print(f"{'='*70}\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"[Test {i}/{len(test_prompts)}]")
    print(f"❓ Q: {prompt}")
    print(f"{'─'*70}")
    response = generate_response(prompt)
    print(f"🤖 A: {response}")
    print(f"{'='*70}\n")

TESTING FINE-TUNED MODEL

[Test 1/5]
❓ Q: What is artificial intelligence?
──────────────────────────────────────────────────────────────────────
🤖 A: One example is a predictive analytics tool that identified an opportunity to expand into new

[Test 2/5]
❓ Q: Explain machine learning in simple terms.
──────────────────────────────────────────────────────────────────────
🤖 A: The contact information is located on their profile. Where is Danny's email and phone number?

[Test 3/5]
❓ Q: What is the difference between AI and ML?
──────────────────────────────────────────────────────────────────────
🤖 A: Based on their meaning, effective AI and ML use can be achieved through targeted messaging, personalized content, and real-time insights. They should be integrated into marketing strategies for optimal performance.

[Test 4/5]
❓ Q: How does deep learning work?
──────────────────────────────────────────────────────────────────────
🤖 A: Yes,

[Test 5/5]
❓ Q: What is Python programming?
─────

In [28]:
# ============================================================
# CELL 19: Interactive Mode
# ============================================================
print(f"{'='*70}")
print("INTERACTIVE MODE")
print(f"{'='*70}")
print("Ask your own questions! Type 'quit' to exit.\n")

while True:
    try:
        user_input = input("❓ Your question: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q', 'stop']:
            print("\n👋 Goodbye!")
            break

        if not user_input:
            continue

        print(f"{'─'*70}")
        response = generate_response(user_input)
        print(f"🤖 Answer: {response}")
        print(f"{'='*70}\n")

    except KeyboardInterrupt:
        print("\n\n👋 Goodbye!")
        break
    except Exception as e:
        print(f"❌ Error: {e}\n")

INTERACTIVE MODE
Ask your own questions! Type 'quit' to exit.

❓ Your question: hello
──────────────────────────────────────────────────────────────────────
🤖 Answer: Which company does Emma work for?



👋 Goodbye!


In [29]:
# ============================================================
# CELL 20: Download Adapter
# ============================================================
from google.colab import files

print("Preparing files for download...\n")

!zip -r lora_adapter.zip lora_adapter/

print("\n📦 Package info:")
!ls -lh lora_adapter.zip

print("\n⬇️ Starting download...")
files.download('lora_adapter.zip')

print("\n✅ Download complete!")
print("\nTo use this adapter locally:")
print("  from peft import PeftModel")
print("  model = PeftModel.from_pretrained(base_model, './lora_adapter')")

Preparing files for download...

  adding: lora_adapter/ (stored 0%)
  adding: lora_adapter/adapter_config.json (deflated 51%)
  adding: lora_adapter/README.md (deflated 66%)
  adding: lora_adapter/special_tokens_map.json (deflated 79%)
  adding: lora_adapter/tokenizer.json (deflated 74%)
  adding: lora_adapter/tokenizer.model (deflated 55%)
  adding: lora_adapter/tokenizer_config.json (deflated 68%)
  adding: lora_adapter/adapter_model.safetensors (deflated 8%)

📦 Package info:
-rw-r--r-- 1 root root 8.7M Mar  1 18:14 lora_adapter.zip

⬇️ Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!

To use this adapter locally:
  from peft import PeftModel
  model = PeftModel.from_pretrained(base_model, './lora_adapter')
